# Episode 19: A Knowledge Base

An agent's prompt is fixed at build time. But real assistants need **durable knowledge** — product facts, policies, documents — that can grow and change without re-writing the prompt.

**In this episode you'll build:** a support agent backed by a knowledge base it consults on every question.

## The pieces

Three things work together, all wired through the `knowledge=` constructor slot:

| Piece | Role |
|---|---|
| `KnowledgeStore` | Path-based storage for knowledge (`MemoryKnowledgeStore`, `DiskKnowledgeStore`, …) |
| `KnowledgeConfig` | Attaches a store to an agent |
| `WorkingMemoryPolicy` | An assembly policy that injects `/memory/working.md` into every prompt |

The store is a tiny filesystem-like key/value space. `WorkingMemoryPolicy` — one of the assembly policies from Episode 8 — reads a known file from it and injects the content as prompt context. The agent simply *always knows* what's in working memory.

## Setup

We write the company knowledge into the store at the path `WorkingMemoryPolicy` reads — `/memory/working.md`.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent, KnowledgeConfig
from autogen.beta.config import OpenAIConfig
from autogen.beta.knowledge import MemoryKnowledgeStore
from autogen.beta.policies import WorkingMemoryPolicy

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

FACTS = """# Company knowledge base

## NovaDB
NovaDB is our database product. It costs $99/month and includes unlimited queries.

## Refund policy
We offer a 30-day money-back guarantee on all plans, no questions asked.
"""

## Seed the store and build the agent

`KnowledgeConfig(store=store)` attaches the store; `assembly=[WorkingMemoryPolicy()]` makes the agent inject `/memory/working.md` on every turn. The agent never has to "search" — the knowledge is simply present.

In [1]:
store = MemoryKnowledgeStore()
await store.write("/memory/working.md", FACTS)

agent = Agent(
    "support",
    prompt="You answer customer questions using your knowledge base.",
    config=config,
    knowledge=KnowledgeConfig(store=store),
    assembly=[WorkingMemoryPolicy()],  # injects /memory/working.md every turn
)

reply = await agent.ask("How much does NovaDB cost and what is your refund policy?")
print("ANSWER:", reply.body)

ANSWER: NovaDB costs $99 per month and includes unlimited queries. Our refund policy offers a 30-day money-back guarantee on all plans, no questions asked.


## What just happened

The agent's prompt never mentioned NovaDB or refunds. `WorkingMemoryPolicy` read `/memory/working.md` from the store and injected it before the LLM call, so the facts were in context.

Change a price? Write the file again — no prompt edit, no redeploy. The knowledge base is **data**, separate from the agent.

## Additional content

**Durable stores.** `MemoryKnowledgeStore()` lives only as long as the process. Swap in `DiskKnowledgeStore(path)` and the knowledge survives restarts — the same agent, rebuilt against the same store, still "remembers".

**Writing memory automatically.** Here we seeded the store by hand. An **aggregate** (`WorkingMemoryAggregate`) can have the agent *write its own* working memory — summarising each conversation into `/memory/working.md` so the next session continues where the last left off. That's the basis of an agent that remembers a user across days.

**Episodic memory.** `EpisodicMemoryPolicy` injects summaries of past *conversations* rather than one rolling file — useful for "what did we discuss last week?" recall.

**It's all assembly.** Knowledge injection is just an assembly policy (Episode 8). It composes with the others — `WorkingMemoryPolicy()` then `SlidingWindowPolicy(...)` gives an agent durable facts *and* a bounded conversation window.

## What's next

This agent's knowledge was loaded ahead of time. Episode 20 lets an agent reach *outside* — fetching live information from the web.